**Gym-based trading environment where the agent learns to buy, sell, or hold stocks while considering risk management.**

**PPO-Based Risk Analysis Model**


Set up the environment: Define a custom Gym environment that includes risk factors like stop-loss, volatility-based position sizing, or reward penalties.


Preprocess stock data: Load historical stock prices and normalize features like moving averages, RSI, MACD, Bollinger Bands.

Train the PPO agent: Use Stable-Baselines3 to train the agent in a reinforcement learning loop.

Implement risk management strategies: Add stop-loss, position sizing, and portfolio constraints to balance risk and reward.

Evaluate and visualize performance: Use Sharpe ratio, VaR (Value at Risk), and Monte Carlo simulations to analyze risk-adjusted returns.

In [1]:
!pip install stable-baselines3


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 90.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 44.0 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstallin

In [2]:
!pip install "shimmy>=2.0" gymnasium


In [3]:
import gym
import numpy as np
import pandas as pd
from gym import spaces
from stable_baselines3 import PPO

#  Create a Custom Trading Environment
class CustomTradingEnv(gym.Env):
    def __init__(self, df):
        super(CustomTradingEnv, self).__init__()

        self.df = df
        self.current_step = 0
        self.initial_balance = 10000  # Starting cash
        self.cash = self.initial_balance
        self.shares_held = 0
        self.max_steps = len(df) - 1

        # Action space: Buy (1), Sell (2), Hold (0)
        self.action_space = spaces.Discrete(3)

        # Observation space: [Current Price, Moving Average, RSI, Cash Available, Shares Held]
        self.observation_space = spaces.Box(
            low=0, high=np.inf, shape=(5,), dtype=np.float32
        )

    def reset(self):
        self.current_step = 0
        self.cash = self.initial_balance
        self.shares_held = 0
        return self._next_observation()

    def _next_observation(self):
        obs = np.array([
            self.df.iloc[self.current_step]["Close"],
            self.df.iloc[self.current_step]["SMA_50"],
            self.df.iloc[self.current_step]["RSI"],
            self.cash,
            self.shares_held
        ])
        return obs

    def step(self, action):
        current_price = self.df.iloc[self.current_step]["Close"]

        if action == 1 and self.cash >= current_price:
            self.shares_held += 1
            self.cash -= current_price
        elif action == 2 and self.shares_held > 0:
            self.shares_held -= 1
            self.cash += current_price

        self.current_step += 1

        done = self.current_step >= self.max_steps
        reward = self.cash + (self.shares_held * current_price) - self.initial_balance

        return self._next_observation(), reward, done, {}



In [5]:
#  Load Stock Data
df = pd.read_csv("/content/drive/MyDrive/preprocessed_stock_data.csv")  # Use actual stock data
df["SMA_50"] = df["Close"].rolling(window=50).mean()
df["RSI"] = 100 - (100 / (1 + df["Close"].pct_change().rolling(14).mean()))


In [6]:
print(df.isna().sum())  # Check for NaNs
df = df.dropna()        # Remove NaNs if present


Date             0
Open             0
High             0
Low              0
Close            0
Volume           0
Ticker           0
MA_10           54
MA_50          294
RSI_14          13
MACD             0
MACD_Signal      0
BB_Upper       114
BB_Lower       114
Close_Norm       0
SMA_50          49
RSI             14
dtype: int64


In [7]:

# Train the PPO Model
env = CustomTradingEnv(df)
model = PPO("MlpPolicy", env, verbose=1)
model.learn(total_timesteps=10000)

#  Evaluate the Model
obs = env.reset()
done = False
while not done:
    action, _ = model.predict(obs)
    obs, reward, done, _ = env.step(action)

print("Final Portfolio Value:", env.cash + (env.shares_held * df.iloc[-1]["Close"]))

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/usr/local/lib/python3.11/dist-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 1.2e+03  |
|    ep_rew_mean     | 1.26e+06 |
| time/              |          |
|    fps             | 755      |
|    iterations      | 1        |
|    time_elapsed    | 2        |
|    total_timesteps | 2048     |
---------------------------------
-------------------------------------------
| rollout/                |               |
|    ep_len_mean          | 1.2e+03       |
|    ep_rew_mean          | 1.83e+06      |
| time/                   |               |
|    fps                  | 626           |
|    iterations           | 2             |
|    time_elapsed         | 6             |
|    total_timesteps      | 4096          |
| train/                  |               |
|    approx_kl            | 4.9388385e-05 |
|    clip_fraction        | 0             |
|    clip_range           | 0.2           |
|    entropy_loss         | -1.1          |
|    explained_variance   | -5.96e-07     |


PPO model's training metrics indicate some issues. The fluctuating ep_rew_mean suggests unstable learning, while the high value_loss means the critic network struggles to predict rewards accurately. The very low approx_kl indicates minimal policy updates, which can slow learning. To improve, consider adjusting hyperparameters like learning rate, reward scaling, and entropy coefficient.

**Improve Risk Awareness in PPO**

current PPO agent likely maximizes profits but doesn't account for risk-adjusted returns. Update your CustomTradingEnv class:

Penalize large drawdowns and volatility.
Reward stable profits and low risk (e.g., using Sharpe ratio).

In [8]:
def _calculate_reward(self):
    # Reward is based on portfolio value but penalizes high risk
    portfolio_return = (self.cash + self.shares_held * self.df.iloc[self.current_step]["Close"]) / self.initial_cash
    sharpe_ratio = np.mean(self.returns) / (np.std(self.returns) + 1e-6)  # Avoid div by zero

    risk_penalty = - abs(self.returns[-1])  # Penalize large fluctuations
    reward = portfolio_return + 0.1 * sharpe_ratio + risk_penalty

    return reward


/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


**Adjust PPO Hyperparameters**

In [9]:
model = PPO("MlpPolicy", env, verbose=1, learning_rate=0.0001, gamma=0.99, gae_lambda=0.95, vf_coef=0.5)


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/usr/local/lib/python3.11/dist-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


In [10]:
model.learn(total_timesteps=100000)


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 1.2e+03  |
|    ep_rew_mean     | 2.25e+06 |
| time/              |          |
|    fps             | 810      |
|    iterations      | 1        |
|    time_elapsed    | 2        |
|    total_timesteps | 2048     |
---------------------------------
-------------------------------------------
| rollout/                |               |
|    ep_len_mean          | 1.2e+03       |
|    ep_rew_mean          | 1.06e+06      |
| time/                   |               |
|    fps                  | 545           |
|    iterations           | 2             |
|    time_elapsed         | 7             |
|    total_timesteps      | 4096          |
| train/                  |               |
|    approx_kl            | 2.1942833e-06 |
|    clip_fraction        | 0             |
|    clip_range           | 0.2           |
|    entropy_loss         | -1.1          |
|    explained_variance   | -7.03e-06     |


In [11]:
from stable_baselines3 import PPO

# Define model save path
model_path = "ppo_stock_trading"

# Load previous model if available
try:
    model = PPO.load(model_path)
    print("Model loaded successfully.")
except:
    print("No existing model found. Training from scratch.")
    model = None

# Define PPO with adjusted hyperparameters
model = PPO(
    "MlpPolicy",
    env,  # Ensure `env` is properly initialized
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.95,
    gae_lambda=0.9,
    clip_range=0.3,
    vf_coef=0.7,
    ent_coef=0.01,
    verbose=1,
    tensorboard_log="./ppo_stock_trading_tensorboard/",
)

# Train the model
model.learn(total_timesteps=1000000)

# Ensure model_path is defined before saving
if model_path:
    model.save(model_path)
    print(f"Model training complete and saved at {model_path}!")


Streaming output truncated to the last 5000 lines.
-----------------------------------------
-------------------------------------------
| rollout/                |               |
|    ep_len_mean          | 1.2e+03       |
|    ep_rew_mean          | 2.57e+06      |
| time/                   |               |
|    fps                  | 505           |
|    iterations           | 252           |
|    time_elapsed         | 1020          |
|    total_timesteps      | 516096        |
| train/                  |               |
|    approx_kl            | 9.1901165e-07 |
|    clip_fraction        | 0             |
|    clip_range           | 0.3           |
|    entropy_loss         | -1.09         |
|    explained_variance   | 0.036         |
|    learning_rate        | 0.0003        |
|    loss                 | 6.06e+08      |
|    n_updates            | 2510          |
|    policy_gradient_loss | -1.78e-05     |
|    value_loss           | 9.2e+08       |
---------------------------

In [12]:
# Save the trained model
model.save(model_path)
print("Model training complete and saved!")

Model training complete and saved!


In [13]:
from google.colab import files

files.download("ppo_stock_trading.zip")  # If you zipped the model
# OR
files.download("ppo_stock_trading.zip")  # If you just saved the model


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**evaluate the model performance**

In [14]:
from stable_baselines3 import PPO

# Define model path
model_path = "ppo_stock_trading.zip"

# Load the trained PPO model
model = PPO.load(model_path)
print("Model Loaded Successfully!")


Model Loaded Successfully!


In [15]:
!pip install finrl


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.2/127.2 kB 2.7 MB/s eta 0:00:00


In [16]:
!pip install alpaca-trade-api

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.8/123.8 kB 5.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 757.7/757.7 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.4/107.4 kB 7.7 MB/s eta 0:00:00
  Created wheel for msgpack: filename=msgpack-1.0.3-cp311-cp311-linux_x86_64.whl size=15688 sha256=ea207ed1ce61230c2b026c2dcd4525b04c16146e71bfaa2722e9671d18bf7760
  Stored in directory: /root/.cache/pip/wheels/f6/35/da/ed9b26b510235e00e3a3c3bab7bad97b59214729662255ab3d
Successfully built msgpack
  Attempting uninstall: msgpack
    Found existing installation: msgpack 1.1.0
    Uninstalling msgpack-1.1.0:
      Successfully uninstalled msgpack-1.1.0
  Attempting uninstall: websockets
    Found existing installation: websockets 14.2
    Uninstalling 

In [17]:
!pip install exchange-calendars

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 198.0/198.0 kB 6.4 MB/s eta 0:00:00


In [18]:
!pip install stockstats

In [19]:
!pip install wrds

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 30.7 MB/s eta 0:00:00


In [20]:
!pip install git+https://github.com/AI4Finance-Foundation/FinRL.git


  Cloning https://github.com/AI4Finance-Foundation/FinRL.git to /tmp/pip-req-build-2u7_u_7q
  Running command git clone --filter=blob:none --quiet https://github.com/AI4Finance-Foundation/FinRL.git /tmp/pip-req-build-2u7_u_7q
  Resolved https://github.com/AI4Finance-Foundation/FinRL.git to commit 3a10805389c07e13e94333e94dcfe9beafce49b0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/AI4Finance-Foundation/ElegantRL.git to /tmp/pip-install-h3xrrs2x/elegantrl_0b0bd8c2a81549de909e2242ac34b3b3
  Running command git clone --filter=blob:none --quiet https://github.com/AI4Finance-Foundation/ElegantRL.git /tmp/pip-install-h3xrrs2x/elegantrl_0b0bd8c2a81549de909e2242ac34b3b3
  Resolved https://github.com/AI4Finance-Foundation/ElegantRL.git to commit 2fa34dd9236498beada8d8443d927970a9de1f7f
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.7/

In [21]:
!pip uninstall -y finrl


Found existing installation: finrl 0.3.6
Uninstalling finrl-0.3.6:
  Successfully uninstalled finrl-0.3.6


In [22]:
!pip install finrl

  Using cached FinRL-0.3.7-py3-none-any.whl.metadata (909 bytes)
Using cached FinRL-0.3.7-py3-none-any.whl (127 kB)


In [23]:
import finrl
print("FinRL installed successfully!")


FinRL installed successfully!


In [24]:
!git clone https://github.com/AI4Finance-Foundation/FinRL.git
%cd FinRL
!pip install .


Cloning into 'FinRL'...


/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


remote: Enumerating objects: 13447, done.
remote: Counting objects: 100% (97/97), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 13447 (delta 77), reused 40 (delta 40), pack-reused 13350 (from 4)
Receiving objects: 100% (13447/13447), 82.51 MiB | 25.86 MiB/s, done.
Resolving deltas: 100% (8960/8960), done.
/content/FinRL
Processing /content/FinRL
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/AI4Finance-Foundation/ElegantRL.git to /tmp/pip-install-7y8y2ao9/elegantrl_c19bcb8f7c7744e38ab3c339a3802ad3
  Running command git clone --filter=blob:none --quiet https://github.com/AI4Finance-Foundation/ElegantRL.git /tmp/pip-install-7y8y2ao9/elegantrl_c19bcb8f7c7744e38ab3c339a3802ad3
  Resolved https://github.com/AI4Finance-Foundation/ElegantRL.git to commit 2fa34dd9236498beada8d8443d927970a9de1f7f
  Preparing metadata (setup.py) ... done
  Created wheel for 

In [26]:
!pip uninstall -y finrl
!pip install --upgrade pip setuptools wheel
